In [ ]:
# load required libraries
import sys
sys.path.append("../utils")
from pairing_utils import iou_batch
from json_parser import parse_json_annotations, show_sample, TestDataSet, OPTICAL_CHARACTERISTICS
from cv_utils import show_detections
from precision_recall_eval import AnnotationFilter, evaluate_yolo_pr, evaluate_mask_rcnn_pr, calculate_confusion_matrix, calculate_confusion_matrix
import numpy as np
import pandas as pd
import os
import cv2
from typing import List, Dict, Union, Tuple, Final
from mask_rcnn_model import run_mask_rcnn, detector, post_process_detections

In [ ]:
print("The model's label map:", detector.get_label_map())

## Evaluation (test) Dataset
### Define filters to aligned dataset labels with the model

In [ ]:
CLASS_IDS_TO_CLASSNAMES_MAP = {1: 'cell', 2: 'bead',  3: 'cage', 4: 'nucleus', 5: 'cell-adhered', 6: 'soma'}
# this should be consistent with the model label map
CLASS_NAMES_TO_CLASS_IDS_MAP = {v: k for k, v in detector.get_label_map().items()}
# different datasets use different names for the same object type, unify them below
# here, we are not filtering any class, that can be done by mapping a class to 'bg' in the filter
CLASS_NAMES_TO_CLASS_IDS_MAP['Cell'] = 1             # in some old datasets -> mapped to cell
CLASS_NAMES_TO_CLASS_IDS_MAP['dying/dead cells'] = 1 # in the old microscope dataset -> mapped to cell
CLASS_NAMES_TO_CLASS_IDS_MAP['dead-cell'] = 1        # in the adhered neuron dataset -> mapped to cell
CLASS_NAMES_TO_CLASS_IDS_MAP['Bead'] = 2             # in some old datasets -> mapped to bead
CLASS_NAMES_TO_CLASS_IDS_MAP['cages'] = 3            # in some old datasets -> mapped to cage
# CLASS_NAMES_TO_CLASS_IDS_MAP['nucleus'] = 4        # we are not interested in checking the model performance on this class, hence ignoring it
CLASS_NAMES_TO_CLASS_IDS_MAP['cytoplasm'] = 5        # in some old datasets -> mapped to cell-adhered
CLASS_NAMES_TO_CLASS_IDS_MAP['cell-adhered'] = 5     # 
CLASS_NAMES_TO_CLASS_IDS_MAP['soma'] = 6             # 

for k, v in detector.get_label_map().items():
    if k not in CLASS_IDS_TO_CLASSNAMES_MAP or v != CLASS_IDS_TO_CLASSNAMES_MAP[k]:
        print(f"[WARN] (class_id, class_name) pair ({k}, {v}) is defined in the model label map but not covered in CLASS_IDS_TO_CLASSNAMES_MAP")


annotation_filter = AnnotationFilter(classnames_mapping_dict = {'soma': 'cell', 'cell-adhered': 'cell'}, 
                                     class_ids_to_class_names_map = CLASS_IDS_TO_CLASSNAMES_MAP)

### Instantiating the dataset class

In [ ]:
dataset = TestDataSet(dataset_paths = [# '/home/cellareye/Cellanome/Data/old_microscope_data_and_old_analysis_data_sets_1_2_test_set',
                                       # '/home/cellareye/Cellanome/Data/imr_90_nucleus_cytoplasm_sets_1_2',
                                       # '/home/cellareye/Cellanome/Data/imr_90_nucleus_cytoplasm_cage_set_3',
                                       # '/home/cellareye/Cellanome/Data/231212_imr90_multichannel_overlay', 
                                       # '/home/cellareye/Cellanome/Data/240213_imr90_multichannel_overlay',
                                       # '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_jurkat_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_k562_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_k562_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_nk92_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_nk92_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240425_nk92_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240425_nk92_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240228_hela-suspension_10x_uncaged', 
                                       # '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_caged', 
                                       # '/home/cellareye/Cellanome/Data/20240314_imr90-suspension_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240307_pbmc-beads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240305_pbmc-nobeads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-beads_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240306_mousepbmc-nobeads_10x_uncaged',  
                                       '/home/cellareye/Cellanome/Data/20240422_neuron-adhered_10x_uncaged', 
                                       # '/home/cellareye/Cellanome/Data/20240509_Hs675Tfibroblasts_10x_caged', 
                                       # '/home/cellareye/Cellanome/Data/20240509_hela-adhered_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240515_DC-adhered_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240516_DC-adhered_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240624_mc38_10x_caged',
                                       # '/home/cellareye/Cellanome/Data/20240624_mc38_10x_uncaged',
                                       # '/home/cellareye/Cellanome/Data/20240625_mc38_10x_caged',
                                       '/home/cellareye/Cellanome/Data/20240703_neuron-adhered_10x_caged', 
                                      ], 
                      scale_factor_dict = {}, 
                      max_larger_side = 5000,
                      max_smaller_side = 5000,
                      class_names_to_ids_map = CLASS_NAMES_TO_CLASS_IDS_MAP,  
                      labels_of_interest = list(CLASS_NAMES_TO_CLASS_IDS_MAP.keys()),
                      min_object_diameter = 6.0)
print(f"Number of samples in the dataset: {len(dataset)}")

### Running the model

In [ ]:
predictions: List = []
run_times: List = []
model_label_map: Dict[int, str] = detector.get_label_map()
for idx in range(len(dataset)):
    annots = dataset[idx]
    preds, run_time = run_mask_rcnn(annots['image'], 
                                    normalize_image = False, 
                                    bit_depth = 8, 
                                    crop = True, 
                                    post_process_class_names = list(model_label_map.values()),
                                    return_features=False,
                                    plot_results = False)
    
    run_times.append(run_time)
    
    # no need to filter for labels_of_interest here as we are doing it during precision and 
    # recall evaluation, similary, no need to only return the classes of interest from the
    # dataset
    
    predictions.append(
        {'boxes': np.array(preds['boxes']) if len(preds['boxes']) > 0 else np.zeros((0, 4), dtype=int), 
         'labels': np.array(preds['labels']) if len(preds['labels']) > 0 else np.zeros((0,), dtype=int), 
         'scores': np.array(preds['scores']) if len(preds['scores']) > 0 else np.zeros((0,), dtype=float), 
         'masks': preds['masks'], })
    
print(f"Running Mask RCNN took {np.mean(run_times[1:]) * 1000}ms on average per image")

In [ ]:
precision = {}
recall = {}
for class_id, class_name in model_label_map.items():
    # use the same function for calculating precision and recall of the Mask RCNN
    # detections if you do not want to use mask IoU (use box IoU)
    # precision, recall = evaluate_yolo_pr(predictions, dataset, class_ids_of_interest, 0.5)
    precision[class_name], recall[class_name] = evaluate_mask_rcnn_pr(predictions, dataset, [class_id], 0.5, None, annotation_filter)
    print(f"Completed PR calculation for '{class_name}' with ID: {class_id}")

for class_name in precision:
    print(f"Mask RCNN Precision: {precision[class_name]}, Recall: {recall[class_name]} at IoU 0.5 for labels: {class_name}")
    if precision[class_name] + recall[class_name] > 0:
        print(f"Mask RCNN F-1 Score: {2 * precision[class_name] * recall[class_name] / (precision[class_name] + recall[class_name])}", 
              f"at IoU 0.5 for labels: {class_name}")

## Confusion matrix
### Bounding boxes or masks
For using masks, set USE_MASKS below to True.

In [ ]:
USE_MASKS = True
dataset_class_ids = list(set(dataset.class_names_to_ids_map.values()))
dataset_class_names = []
ids_predicted_by_model = []
for class_id in dataset_class_ids:
    dataset_class_names.append(CLASS_IDS_TO_CLASSNAMES_MAP[class_id])
    if class_id in model_label_map:
        ids_predicted_by_model.append(class_id)

In [ ]:
c_m = calculate_confusion_matrix(predictions, dataset, USE_MASKS, 0.5)
# confusion matrix is calculated for all the class IDs present in the dataset (dataset_class_ids)
# reshape the confusion matrix to remove class IDs not included in the model's label_map
c_m = c_m[np.array(ids_predicted_by_model + [c_m.shape[0]]) - 1, :]

In [ ]:
c_m_df = pd.DataFrame(index = list(detector.get_label_map().values()) + ['FN'], 
                      columns = dataset_class_names + ['FP'], data=c_m)
c_m_df

In [ ]:
# in the following, if there are multiple dataset class IDs that are mapped to one, we use them to compute
# precision and recall correctly
model_class_id_to_dataset_class_ids_map = {}
for class_id in model_label_map:
    mapped_ids = [class_id]
    for k, v in annotation_filter.class_ids_mapping_dict.items():
        if v == class_id:
            mapped_ids.append(dataset.class_names_to_ids_map[CLASS_IDS_TO_CLASSNAMES_MAP[k]])
    mapped_ids = list(set(mapped_ids))
    model_class_id_to_dataset_class_ids_map[class_id] = [CLASS_IDS_TO_CLASSNAMES_MAP[k] for k in mapped_ids]   

In [ ]:
precision = {}
recall = {}
recall_w_break_down = {}
for class_id in model_label_map:
    row = c_m_df.loc[CLASS_IDS_TO_CLASSNAMES_MAP[class_id], model_class_id_to_dataset_class_ids_map[class_id] + ['FP']].values
    if np.sum(row) > 0:
        precision[CLASS_IDS_TO_CLASSNAMES_MAP[class_id]] = np.round(np.sum(row[:-1]) / np.sum(row), 3)
    fn = c_m_df.loc['FN', model_class_id_to_dataset_class_ids_map[class_id]].values.sum()
    if np.sum(row[:-1]) + fn > 0:
        recall[CLASS_IDS_TO_CLASSNAMES_MAP[class_id]] = np.round(np.sum(row[:-1]) / (np.sum(row[:-1]) + fn), 3)
    

for class_id in dataset_class_ids:
    if class_id in annotation_filter.class_ids_mapping_dict:
        model_class_id = annotation_filter.class_ids_mapping_dict[class_id]
    else:
        model_class_id = class_id
    tp = c_m_df.loc[CLASS_IDS_TO_CLASSNAMES_MAP[model_class_id],  CLASS_IDS_TO_CLASSNAMES_MAP[class_id]]
    fn = c_m_df.loc['FN',  CLASS_IDS_TO_CLASSNAMES_MAP[class_id]]
    if tp + fn > 0:
        recall_w_break_down[CLASS_IDS_TO_CLASSNAMES_MAP[class_id]] = np.round(tp / (tp + fn), 3)
    else:
        recall_w_break_down[CLASS_IDS_TO_CLASSNAMES_MAP[class_id]] = 0

In [ ]:
p_r_with_break_down_df = pd.DataFrame(data=[precision, recall_w_break_down], index = ['Precision', 'Recall']).transpose()
p_r_with_break_down_df

In [ ]:
p_r_df = pd.DataFrame(data=[precision, recall], index = ['Precision', 'Recall']).transpose()
p_r_df

In [ ]:
from PIL import Image
idx = 5
annots = dataset[idx]
img = show_sample(sample=annots, class_id_to_name_mapping=mask_rcnn_model.detector.get_label_map())
Image.fromarray(img)

In [ ]:
display(Image.fromarray(annots['image']))

In [ ]:
cv2.imwrite('sample.jpg', annots['image'])

In [ ]:
img = show_detections(input_image=annots['image'], predictions=predictions[idx], label_map=mask_rcnn_model.detector.get_label_map())
Image.fromarray(img)

In [ ]:
"""
### All datasets excluding two IMR-90 adhered sets imr_90_nucleus_cytoplasm_sets_1_2 and imr_90_nucleus_cytoplasm_cage_set_3

Existing)
Mask RCNN Precision: 0.7828456772885704, Recall: 0.8479834192247367 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8141136990647874 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9462918004542251, Recall: 0.9469889338223769 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9466402387910031 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9777642770352369, Recall: 0.9954230578921326 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9865146499938703 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7641242937853108, Recall: 0.6728855721393034 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7156084656084656 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7453505007153076, Recall: 0.8052550231839258 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.774145616641902 at IoU 0.5 for labels: cell-adhered

14)
Mask RCNN Precision: 0.8932711447640608, Recall: 0.9428594969823058 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.917395704937418 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9715778422157785, Recall: 0.9640111116622848 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9677796867607878 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9689405758048504, Recall: 0.9950480629187299 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9818207947115039 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7988826815642458, Recall: 0.7114427860696517 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7526315789473683 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8151001540832049, Recall: 0.8176197836166924 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.816358024691358 at IoU 0.5 for labels: cell-adhered

18)
Mask RCNN Precision: 0.9005086145686919, Recall: 0.9352566707300556 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9175537805076112 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9768932038834951, Recall: 0.969289457459926 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9730764767320412 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9666186359269933, Recall: 0.9957941613062841 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9809895198635146 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7895424836601307, Recall: 0.7512437810945274 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.769917144678139 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7647887323943662, Recall: 0.839258114374034 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8002947678703021 at IoU 0.5 for labels: cell-adhered

20)
Mask RCNN Precision: 0.8931827103415187, Recall: 0.9176786914631414 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.905265019229919 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9769225932214047, Recall: 0.9740744223956864 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9754964288517385 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9648259303721488, Recall: 0.9941860465116279 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9792859753868648 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7776332899869961, Recall: 0.7437810945273632 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7603305785123967 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7457865168539326, Recall: 0.8207109737248841 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7814569536423841 at IoU 0.5 for labels: cell-adhered

21)
Mask RCNN Precision: 0.8910552327160158, Recall: 0.919056688326238 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.904839376809596 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9831337894336754, Recall: 0.9730555816796764 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9780687242992863 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9906056860321384, Recall: 0.9913409203364671 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9909731668109311 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8141210374639769, Recall: 0.7027363184079602 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7543391188251001 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8288854003139717, Recall: 0.8160741885625966 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.822429906542056 at IoU 0.5 for labels: cell-adhered

26)
Mask RCNN Precision: 0.9089054334598039, Recall: 0.9257719931459404 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9172611844600574 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9838687296190939, Recall: 0.975867961966807 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9798520139348748 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9917285459159696, Recall: 0.993128765060241 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9924281615952594 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8262773722627738, Recall: 0.7039800995024875 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7602417730020148 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8143525741029641, Recall: 0.8068006182380216 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.81055900621118 at IoU 0.5 for labels: cell-adhered

28)
Mask RCNN Precision: 0.9032705193479246, Recall: 0.9237700100266224 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.913405262006345 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9852167666890655, Recall: 0.9753518299746522 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9802594797724509 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9923532089212562, Recall: 0.991991991991992 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9921725675798672 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7164068299925761, Recall: 0.6678200692041523 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.6912607449856734 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7430262045646661, Recall: 0.7670157068062827 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.75483039931301 at IoU 0.5 for labels: cell-adhered

Mask RCNN Precision: 0.8095238095238095, Recall: 0.9797775530839231 at IoU 0.5 for labels: soma
Mask RCNN F-1 Score: 0.8865507776761208 at IoU 0.5 for labels: soma

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9046040416179976, Recall: 0.9248036224581361 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9145923142298359 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9862384259259259, Recall: 0.9764848790438101 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9813374178725461 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9917310313493867, Recall: 0.9924524870419206 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9920916280338151 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7817286652078774, Recall: 0.9139750559641829 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8426949727259325 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
* recall 0.919/0.892/0.937 for cell/cell-adhered/soma 
Mask RCNN Precision: 0.8902129943446672, Recall: 0.9176946721311475 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9037449612285773 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9861922648859209, Recall: 0.9748117758958551 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9804689976313833 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9919732055205867, Recall: 0.9936946838664893 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.992833198474165 at IoU 0.5 for labels: cage


### Jurkat sets only

Existing)
Mask RCNN Precision: 0.8977272727272727, Recall: 0.9320224719101123 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.914553472987872 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9747340425531915, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9872053872053873 at IoU 0.5 for labels: cage

14)
Mask RCNN Precision: 0.9520868425977915, Recall: 0.952621722846442 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9523542076195826 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9644736842105263, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9819156061620896 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.9564971751412429, Recall: 0.951123595505618 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9538028169014084 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9632063074901446, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9812583668005356 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.9599171998494542, Recall: 0.9552434456928839 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.957574619861085 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9619422572178478, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9806020066889632 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.9567434831885153, Recall: 0.9485018726591761 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9526048523603536 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9972789115646259, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9986376021798365 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.9572343632253203, Recall: 0.951498127340824 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9543576258452292 at IoU 0.5 for labels: cell


Mask RCNN Precision: 0.9972789115646259, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9986376021798365 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9583804143126177, Recall: 0.9521047708138447 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9552322853120601 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.9583018867924529, Recall: 0.9502338634237605 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9542508219821513 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9986376021798365, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9993183367416496 at IoU 0.5 for labels: cage


### K562 sets only

Existing)
Mask RCNN Precision: 0.815254652301665, Recall: 0.9352528089887641 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8711407639979069 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.96271637816245, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9810040705563093 at IoU 0.5 for labels: cage

14)
Mask RCNN Precision: 0.9031461258450338, Recall: 0.9757022471910113 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9380232244126384 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.96271637816245, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9810040705563093 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.924963151547635, Recall: 0.9695224719101123 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9467187821435918 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.965287049399199, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9823369565217391 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.9267771245323356, Recall: 0.9741573033707865 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9498767460969597 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.964, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9816700610997963 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.9142630325482932, Recall: 0.9705056179775281 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9415451696416405 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9986149584487535, Recall: 0.9972337482710927 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9979238754325259 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.9167438782263402, Recall: 0.972752808988764 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9439182282793867 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9986149584487535, Recall: 0.9972337482710927 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9979238754325259 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9198884017536867, Recall: 0.9712442137747229 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9448689956331877 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9972375690607734, Recall: 0.9972375690607734 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9972375690607734 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.9170421415319374, Recall: 0.9706831252630103 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9431005110732539 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9986149584487535, Recall: 0.9958563535911602 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9972337482710927 at IoU 0.5 for labels: cage


### NK92 sets only

Existing)
Mask RCNN Precision: 0.5662620961128424, Recall: 0.8128310771041789 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.6675044709749143 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.975609756097561, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9876543209876543 at IoU 0.5 for labels: cage

14)
Mask RCNN Precision: 0.8207984119982356, Recall: 0.8761624484991172 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8475772931731481 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9549071618037135, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9769335142469471 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.8455813953488373, Recall: 0.8560329605650383 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8507750804328752 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9612817089452603, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9802586793737236 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.8654481132075472, Recall: 0.8639199529134786 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8646833578792342 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9574468085106383, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9782608695652174 at IoU 0.5 for labels: cage

21)
Mask RCNN Precision: 0.8335208098987626, Recall: 0.872277810476751 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8524590163934426 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.8418827686289356, Recall: 0.8851077943615258 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8629543396714526 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.8532351057014734, Recall: 0.8835157545605307 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8681114551083591 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.851123416714404, Recall: 0.8846840886536553 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8675793161616817 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.8503097260361453, Recall: 0.8809130003307972 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8653408721648145 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9994663820704376, Recall: 0.9994663820704376 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9994663820704376 at IoU 0.5 for labels: cage


### HeLa suspension sets only

Existing)
Mask RCNN Precision: 0.9494403705133153, Recall: 0.9574468085106383 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9534267812156838 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9822380106571936, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9910394265232976 at IoU 0.5 for labels: cage

14)
Mask RCNN Precision: 0.9672260094389092, Recall: 0.9571873378308251 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9621804903495045 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9804964539007093, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.990152193375112 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.9704991439483736, Recall: 0.9560197197716658 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9632050192797857 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9787610619469026, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9892665474060822 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.9679453494482396, Recall: 0.9558899844317592 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9618798955613578 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9753086419753086, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9875 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.9746448957918492, Recall: 0.9525168655941879 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9634538416114428 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.9753903148981212, Recall: 0.9564089257913856 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9658063670902659 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9798461946433307, Recall: 0.9540407952491609 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9667713239141811 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9981949458483754, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.999096657633243 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.9796974522292994, Recall: 0.953137103020914 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9662347860227718 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage


### IMR-90 suspension sets only

Existing)
Mask RCNN Precision: 0.5912784935579782, Recall: 0.7961035495062717 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.6785714285714286 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9805825242718447, Recall: 0.9975308641975309 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.988984088127295 at IoU 0.5 for labels: cage

14)
Mask RCNN Precision: 0.70188041411367, Recall: 0.8865759274085936 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7834905660377358 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9665871121718377, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9830097087378641 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.7945998071359691, Recall: 0.8796370429677075 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8349588347055099 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9688995215311005, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9842041312272175 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.8, Recall: 0.8871096877502002 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8413059984813971 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9688995215311005, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9842041312272175 at IoU 0.5 for labels: cage

21)
Mask RCNN Precision: 0.7689983732279805, Recall: 0.8831064851881505 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8221118012422359 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9975124378109452, Recall: 0.9901234567901235 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9938042131350681 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.772653156300955, Recall: 0.885241526554577 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8251243781094527 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9975124378109452, Recall: 0.9901234567901235 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9938042131350681 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.7890848427073404, Recall: 0.8836402455297572 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8336900415460154 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9975124378109452, Recall: 0.9901234567901235 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9938042131350681 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.7828962910465391, Recall: 0.884204909284952 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8304723718832228 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 0.9901477832512315 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.995049504950495 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.7730463266261114, Recall: 0.8815368196371398 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8237347294938918 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 0.9901477832512315 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.995049504950495 at IoU 0.5 for labels: cage


### PBMC (mouse and human) sets only

Existing)
Mask RCNN Precision: 0.7580085641498184, Recall: 0.7566141860087648 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7573107332394671 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.8993072942747454, Recall: 0.9178449744463373 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9084815782817637 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9735576923076923, Recall: 0.996309963099631 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9848024316109423 at IoU 0.5 for labels: cage


14)
Mask RCNN Precision: 0.7101891729815188, Recall: 0.9204408959231258 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8017602166420482 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9545793000744601, Recall: 0.8852062834455378 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9185848634124496 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9713261648745519, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9854545454545455 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.9288224350582237, Recall: 0.9524482441885113 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9404869880694899 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9810159055926116, Recall: 0.9901605385810461 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9855670103092784 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9678571428571429, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.983666061705989 at IoU 0.5 for labels: cage

20)
Mask RCNN Precision: 0.888294584061282, Recall: 0.894037764432181 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8911569211686509 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9825502788575291, Recall: 0.9904173764906303 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9864681428692627 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9631828978622328, Recall: 0.997539975399754 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.980060422960725 at IoU 0.5 for labels: cage

21)
Mask RCNN Precision: 0.8889217293172474, Recall: 0.8949575285397392 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8919294177911381 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9911734606856558, Recall: 0.9899914821124361 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9905821188101935 at IoU 0.5 for labels: bead

Mask RCNN Precision: 1.0, Recall: 0.991389913899139 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9956763434218654 at IoU 0.5 for labels: cage

26)
Mask RCNN Precision: 0.925096564733661, Recall: 0.9210307776826666 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9230591940997964 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9889433050321863, Recall: 0.9866222876528196 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9877814329046866 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9986313868613139, Recall: 0.9954524783992724 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9970393987702119 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.9283408567717376, Recall: 0.9222445094607198 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9252826415771528 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9896315338474722, Recall: 0.9856408286927394 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9876321499962587 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9986313868613139, Recall: 0.9954524783992724 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9970393987702119 at IoU 0.5 for labels: cage

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9312316866293117, Recall: 0.9238821384401575 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9275423538974547 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9919287503478987, Recall: 0.9869421663648951 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9894291754756871 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9986313868613139, Recall: 0.9954524783992724 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9970393987702119 at IoU 0.5 for labels: cage

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.9267838809034907, Recall: 0.9207456235417119 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9237548848424378 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9916191537703091, Recall: 0.9854723612738311 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9885362023098537 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9981760145918832, Recall: 0.9954524783992724 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9968123861566485 at IoU 0.5 for labels: cage


### Neurons adhered set only

28)
Mask RCNN Precision: 0.762161499816199, Recall: 0.8750703432751828 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.814722640644443 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.8292682926829268, Recall: 0.9797775530839231 at IoU 0.5 for labels: soma
Mask RCNN F-1 Score: 0.8982618771726535 at IoU 0.5 for labels: soma

28) - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.7606837606837606, Recall: 0.875737981445038 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8141662310507056 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.8274828767123288, Recall: 0.9757698132256436 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.895529302756544 at IoU 0.5 for labels: cell-adhered

31) - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.7639025590551181, Recall: 0.8727860556648861 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8147224773651751 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.8372395833333334, Recall: 0.9737506309944473 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.9003500583430571 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
* recall 0.873/0.937 for cell/soma 
Mask RCNN Precision: 0.7509902703534208, Recall: 0.8920968478757424 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8154845175704174 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 0.999034749034749 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9995171414775471 at IoU 0.5 for labels: cage


### Hs675T Fibroblasts adhered set only

28)
Mask RCNN Precision: 0.5586319218241043, Recall: 0.9475138121546961 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7028688524590164 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9864130434782609, Recall: 0.9945205479452055 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.990450204638472 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.631578947368421, Recall: 0.6177847113884556 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.6246056782334385 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.6776061776061776, Recall: 0.7034068136272545 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.6902654867256637 at IoU 0.5 for labels: cell-adhered


28) - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.6039076376554174, Recall: 0.9392265193370166 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7351351351351353 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9811320754716981, Recall: 0.994535519125683 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9877883310719132 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.6055646481178396, Recall: 0.7414829659318637 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.6666666666666666 at IoU 0.5 for labels: cell-adhered

31) - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.5919439579684763, Recall: 0.9337016574585635 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7245444801714898 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9811827956989247, Recall: 0.9972677595628415 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9891598915989159 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.5957792207792207, Recall: 0.7354709418837675 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.6582959641255606 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
* recall 0.834/0.745 for cell/cell-adhered 
Mask RCNN Precision: 0.6322701688555347, Recall: 0.7828106852497096 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.6995329527763362 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.981081081081081, Recall: 0.9918032786885246 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9864130434782609 at IoU 0.5 for labels: cage

### IMR-90 adhered sets only

28)
Mask RCNN Precision: 0.5553470919324578, Recall: 0.8888888888888888 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.6836027713625866 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9814020028612304, Recall: 0.9554317548746518 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9682427664079041 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.81169757489301, Recall: 0.7077114427860697 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.7561461794019935 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8186046511627907, Recall: 0.8160741885625966 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8173374613003096 at IoU 0.5 for labels: cell-adhered

28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.5938775510204082, Recall: 0.8738738738738738 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.707168894289186 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9914651493598862, Recall: 0.9640387275242047 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9775596072931276 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8125915080527086, Recall: 0.8578052550231839 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8345864661654137 at IoU 0.5 for labels: cell-adhered

31 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.5636704119850188, Recall: 0.9039039039039038 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.6943483275663207 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9872521246458924, Recall: 0.9640387275242047 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9755073477956613 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8313253012048193, Recall: 0.8531684698608965 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8421052631578948 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
* recall 0.877/0.879 for cell/cell-adhered 
Mask RCNN Precision: 0.7506538796861377, Recall: 0.8785714285714286 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8095909732016925 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9858757062146892, Recall: 0.9654218533886584 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9755415793151643 at IoU 0.5 for labels: cage

### HeLa adhered set only

28)
Mask RCNN Precision: 0.3158866995073892, Recall: 0.6812749003984063 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.43163651661758523 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9769392033542977, Recall: 0.966804979253112 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9718456725755995 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.91005291005291, Recall: 0.5403141361256545 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.6780551905387648 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8634615384615385, Recall: 0.5050618672665916 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.6373314407381121 at IoU 0.5 for labels: cell-adhered

31 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.3626760563380282, Recall: 0.6839309428950863 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.4739990796134377 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9688149688149689, Recall: 0.966804979253112 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9678089304257529 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7957446808510639, Recall: 0.6310461192350956 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7038895859473023 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
- recall 0.816/0.874 for cell/cell-adhered
Mask RCNN Precision: 0.8666666666666667, Recall: 0.8471376370280146 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8567908838928241 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9342629482071713, Recall: 0.9730290456431535 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9532520325203252 at IoU 0.5 for labels: cage

### mutuDC sets only 
This set is suffering from cell and cell-adhered classes getting mixed

31 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.8555920391342223, Recall: 0.8798842257597684 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8675681185024122 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9986541049798116, Recall: 0.9995509654243376 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9991023339317774 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.740416367552046, Recall: 0.824065196548418 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7800045375482115 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
- recall 0.903/0.912 for cell/cell-adhered
Mask RCNN Precision: 0.9460480580323409, Recall: 0.9068958856480587 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9260583347715673 at IoU 0.5 for labels: cell

Mask RCNN Precision: 1.0, Recall: 0.9982070820259973 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9991027366532077 at IoU 0.5 for labels: cage

### MC38 sets only

31 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.8864380869185412, Recall: 0.9641330166270784 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9236545682102629 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9899109792284867, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9949299135102893 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.6018957345971564, Recall: 0.7134831460674157 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.6529562982005142 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
- recall 0.945/0.866 for cell/cell-adhered
Mask RCNN Precision: 0.8837726262305494, Recall: 0.9235108677617389 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9032048681541582 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9925665101721439, Recall: 0.9984258166076347 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9954875416911908 at IoU 0.5 for labels: cage

### All adhered datasets
28)
Mask RCNN Precision: 0.7374301675977654, Recall: 0.8789857856319632 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8020096979610913 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9868913857677902, Recall: 0.967860422405877 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9772832637923041 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7233883058470765, Recall: 0.6678200692041523 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.6944944224541202 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7538593481989708, Recall: 0.7670157068062827 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7603806228373703 at IoU 0.5 for labels: cell-adhered

Mask RCNN Precision: 0.8121338912133891, Recall: 0.9798081776880363 at IoU 0.5 for labels: soma
Mask RCNN F-1 Score: 0.8881262868908716 at IoU 0.5 for labels: soma


28 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.7422914638104512, Recall: 0.8786016135228583 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8047149894440535 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9878957169459963, Recall: 0.9742883379247016 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.981044845122515 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.78732782369146, Recall: 0.9139750559641829 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.845937546248335 at IoU 0.5 for labels: cell-adhered

31 - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.8239081259787715, Recall: 0.8923859781379571 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8567809644440424 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.992589625475666, Recall: 0.9943820224719101 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9934850155357322 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.7528028438610883, Recall: 0.8478595626732368 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7975086906141367 at IoU 0.5 for labels: cell-adhered

35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
* Excluding neurons - recall 0.915/0.892 for cell/cell-adhered
Mask RCNN Precision: 0.905501171928857, Recall: 0.9060008277003725 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9057509309060819 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9891560584629892, Recall: 0.9922749487624153 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9907130489532504 at IoU 0.5 for labels: cage

* Including neurons - recall 0.898/0.892/0.937 for cell/cell-adhered/soma
Mask RCNN Precision: 0.8409777092567611, Recall: 0.9007654597058571 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.8698454357449565 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9906731549067316, Recall: 0.9932240140940507 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9919469445760304 at IoU 0.5 for labels: cage

### All suspension datasets

14)
Mask RCNN Precision: 0.8389600521059488, Recall: 0.9351725806939528 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.884457478005865 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9682652317032306, Recall: 0.9464203144266338 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9572181571947738 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9729478848959024, Recall: 0.9960629921259843 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9843697591735426 at IoU 0.5 for labels: cage

18)
Mask RCNN Precision: 0.9038627362144308, Recall: 0.9354297123150921 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9193753391260397 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9770070299452364, Recall: 0.969289457459926 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9731329426101086 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9699894235854045, Recall: 0.9960629921259843 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9828533154722036 at IoU 0.5 for labels: cage

28)
Mask RCNN Precision: 0.9161116683142241, Recall: 0.9262977419472513 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9211765474108847 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9860848810668644, Recall: 0.9753045391517596 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9806650842301754 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9936472723605929, Recall: 0.9945498587000404 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9940983606557378 at IoU 0.5 for labels: cage

31) - 4 class - no nucleus, soma and cell-adhered combined)
Mask RCNN Precision: 0.9149733661061175, Recall: 0.9261080509247437 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9205070377595196 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9850737627087242, Recall: 0.9748576143382648 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9799390626709903 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9925515853044791, Recall: 0.9952563584981833 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9939021317341128 at IoU 0.5 for labels: cage


35 - 3 class - no nucleus, cell, soma and cell-adhered combined)
Mask RCNN Precision: 0.9122756516357461, Recall: 0.9248750592784445 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9185321513062318 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9863294837901767, Recall: 0.9748117758958551 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9805368083132092 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9929428369795342, Recall: 0.9940452159870812 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.993493720683916 at IoU 0.5 for labels: cage

## Precision and Recall calculated for different models and over different test sets

This section documents the precision and recall values for all the Mask R-CNN models trained so far. For each model identified by a name, the list of classnames the model is supposed to detect, the datasets the model is trained on, as well as the datasets the model is test on are provided.

For the datasets, the indexes used in the table are defined below:

| Set index | Processed dataset name | Darwin dataset names | Annotated classes | Crop overlaps x, y | Comments |
|:----------|:----------|:----------|:----------|:----------|:----------|
| 1 | `old_microscope_data_176_160` |  `D1` |  'Cell'<br>'Bead' <br> 'dying/dead cells' <br> 'Cluster'| 176, 160| Old ix-81 microscope dataset|
| 2 | `old_analysis_data_set_1_176_160` |  `Normalised_cytokine_PBMC_10312022` <br> `Normalised_cytokine_NK_10312022` <br> `Normalised_surface_Jurkat_10312022` <br> `Normalised_surface_NK_Jurkat_10312022` <br> `Normalised_surface_PBMC_10312022` |  'Cell'<br>'Bead' <br> 'cages'| 176, 160| Mix of ix-81 and BB2|
| 3 | `old_analysis_data_set_2_176_160` | `Normalised_12212022_tregs_beads_cages_BB2` |  'Cell'<br>'Bead' <br> 'cages'| 176, 160| Old BB2|
| 4 | `imr_90_nucleus_cytoplasm_sets_1_2_458_416`| `230607_IMR90_training_dataset_1_cytoplasm` <br> `230607_IMR90_training_dataset_1_nuclei` <br> `230622_IMR90_training_dataset_3_cytoplasm` <br> `230622_IMR90_training_dataset_3_nuclei`| 'nucleus' <br> 'cytoplasm' | 458, 416 | Early versions of multi-channel datasets annotated separately|
| 5 | `imr_90_nucleus_cytoplasm_cage_set_3_458_416`| `230915_IMR90_training_dataset_3_cytoplasm` <br> `230915_IMR90_training_dataset_3_nuclei` <br> `230915_IMR90_training_dataset_3_cages_bf`| 'nucleus' <br> 'cytoplasm' <br> 'cages'| 458, 416 | Early versions of multi-channel datasets annotated separately|
| 6 |`imr_90_cell_nucleus_cytoplasm_cage_sets_4_5_458_416`| `231212_imr90_multichannel_overlay`| 'cell' <br> 'bead' <br> 'cage' <br> 'nucleus' <br> 'cell-adhered' | 458, 416 | annotated as multi-channel overlaid images|
| 7 | `imr_90_cell_nucleus_cytoplasm_cage_set_6_458_416`|`240213_imr90_multichannel_overlay` | 'cell' <br> 'bead' <br> 'cage' <br> 'nucleus' <br> 'cell-adhered' | 458, 416 | annotated as multi-channel overlaid images|


| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_2.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3, 4, 5   |'Step LR'|

* Mask RCNN Precision: 0.8997133278266363, Recall: 0.9471126796583295 at IoU 0.5 for labels: cell
* Mask RCNN F-1 Score: 0.922804744343666 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.9779136165823963, Recall: 0.9642095342030854 at IoU 0.5 for labels: bead
* Mask RCNN F-1 Score: 0.9710132257621361 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.9869942196531792, Recall: 0.9915795586527294 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.9892815758980302 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.8980988593155893, Recall: 0.7915549597855228 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8414677591734948 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.47413793103448276, Recall: 0.3810623556581986 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4225352112676057 at IoU 0.5 for labels: cell-adhered

| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_1cl_lrs' |1, 2, 3, 4, 5, 6 | 1, 2, 3, 4, 5   |'1-Cycle LR'|


* Mask RCNN Precision: 0.9034226844392363, Recall: 0.9463966037542837 at IoU 0.5 for labels: cell
* Mask RCNN F-1 Score: 0.9244104716227018 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.9774243072508998, Recall: 0.9632422243166824 at IoU 0.5 for labels: bead
* Mask RCNN F-1 Score: 0.9702814455784438 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.9821736630247269, Recall: 0.991869918699187 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.9869979774631609 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.9031273836765827, Recall: 0.7935656836461126 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8448091330717089 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.5004703668861712, Recall: 0.4095458044649731 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4504657070279424 at IoU 0.5 for labels: cell-adhered

| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_2.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3  |'Step LR'|

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.8518518518518519, Recall: 0.92 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.8846153846153846 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.8980988593155893, Recall: 0.7915549597855228 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8414677591734948 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.4755043227665706, Recall: 0.3810623556581986 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.42307692307692313 at IoU 0.5 for labels: cell-adhered


| Model name | Training sets | LR scheduler | Test sets |
|:----------:|:-------------:|:------------:|:---------:|
|'cell_bead_cage_nucl_cyto_mix_crop_0p1_bbox_0p7_1_rs_0p25_blur_2_bs_8_epochs_1cl_lrs.pt' |1, 2, 3, 4, 5, 6 | 1, 2, 3  |'1-Cycle LR'|

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: cell

* Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

* Mask RCNN Precision: 0.8518518518518519, Recall: 0.92 at IoU 0.5 for labels: cage
* Mask RCNN F-1 Score: 0.8846153846153846 at IoU 0.5 for labels: cage

* Mask RCNN Precision: 0.9031273836765827, Recall: 0.7935656836461126 at IoU 0.5 for labels: nucleus
* Mask RCNN F-1 Score: 0.8448091330717089 at IoU 0.5 for labels: nucleus

* Mask RCNN Precision: 0.5009416195856874, Recall: 0.4095458044649731 at IoU 0.5 for labels: cell-adhered
* Mask RCNN F-1 Score: 0.4506565014824227 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei and cell-adhered
Overlaid set (IMR-90 nuclei set 4, 5) test

Mask RCNN Precision: 0.6796875, Recall: 0.8743718592964824 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7648351648351649 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9541062801932367, Recall: 0.9875 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9705159705159706 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8928571428571429, Recall: 0.8566978193146417 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8744038155802861 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7420289855072464, Recall: 0.8737201365187713 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8025078369905956 at IoU 0.5 for labels: cell-adhered

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.7049180327868853, Recall: 0.864321608040201 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7765237020316028 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9541062801932367, Recall: 0.9875 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9705159705159706 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.846875, Recall: 0.8442367601246106 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8455538221528862 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8118811881188119, Recall: 0.8395904436860068 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.825503355704698 at IoU 0.5 for labels: cell-adhered

New cropping Step LRS
Mask RCNN Precision: 0.7160493827160493, Recall: 0.8743718592964824 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.7873303167420813 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9588377723970944, Recall: 0.99 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.974169741697417 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8679245283018868, Recall: 0.8598130841121495 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.863849765258216 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.8066666666666666, Recall: 0.825938566552901 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.8161888701517707 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei and cell-adhered
Combined set (caging set + analysis set 1 + analysis set 2 + IMR-90 set 1, 2, 3, 4, 5 with 4, 5 
being the overlaid images) test

Mask RCNN Precision: 0.8839408773347275, Recall: 0.9417215189873418 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9119168444019514 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9815585349565932, Recall: 0.9478644774046332 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9644173017715642 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9879454926624738, Recall: 0.9807492195629552 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9843342036553524 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.9136735979836169, Recall: 0.7997793712079426 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8529411764705881 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.47537091988130564, Recall: 0.503140703517588 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.4888617638083613 at IoU 0.5 for labels: cell-adhered

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.9010856453558505, Recall: 0.9455696202531646 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9227918468190243 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9773997080585897, Recall: 0.9632422243166824 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9702693249387898 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9791880781089414, Recall: 0.9914151925078044 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9852637021716649 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8920907418761496, Recall: 0.8025372311086597 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8449477351916376 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5695461200585652, Recall: 0.4886934673366834 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5260311020960108 at IoU 0.5 for labels: cell-adhered

New cropping Step LRS
Mask RCNN Precision: 0.8975701114099116, Recall: 0.946379746835443 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9213289298565583 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.9778644195698655, Recall: 0.9642095342030854 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9709889725625366 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9839917376710561, Recall: 0.9914151925078044 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9876895166515485 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8922229026331905, Recall: 0.8036403750689465 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8456181079512478 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5483630952380952, Recall: 0.4629396984924623 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5020435967302452 at IoU 0.5 for labels: cell-adhered

Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
Overlaid set (IMR-90 nuclei set 4, 5) test

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.678030303030303, Recall: 0.8994974874371859 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.773218142548596 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.0, Recall: 0.0 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.948780487804878, Recall: 0.9725 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9604938271604938 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.865814696485623, Recall: 0.8442367601246106 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8548895899053628 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.7745098039215687, Recall: 0.8088737201365188 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.7913188647746243 at IoU 0.5 for labels: cell-adhered


Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
Combined set (caging set + analysis set 1 + analysis set 2 + IMR-90 set 1, 2, 3, 4, 5 with 4, 5 
being the overlaid images) test

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.8996539792387543, Recall: 0.9478481012658228 at IoU 0.5 for labels: cell
Mask RCNN F-1 Score: 0.9231224419350066 at IoU 0.5 for labels: cell

Mask RCNN Precision: 0.97728871470204, Recall: 0.9648296046430874 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9710191957265171 at IoU 0.5 for labels: bead

Mask RCNN Precision: 0.9837167226673559, Recall: 0.9901144640998959 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 0.9869052249448982 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.9001233045622689, Recall: 0.8052950910093767 at IoU 0.5 for labels: nucleus
Mask RCNN F-1 Score: 0.8500727802037846 at IoU 0.5 for labels: nucleus

Mask RCNN Precision: 0.5508537490720119, Recall: 0.46608040201005024 at IoU 0.5 for labels: cell-adhered
Mask RCNN F-1 Score: 0.5049336509016672 at IoU 0.5 for labels: cell-adhered

Combined model for cells, beads, cages, nuclei, cell-adhered and hela cells (as a separate class)
hela cells test dataset

New cropping 1-Cycle LRS
Mask RCNN Precision: 0.99185667752443, Recall: 0.9967266775777414 at IoU 0.5 for labels: bead
Mask RCNN F-1 Score: 0.9942857142857142 at IoU 0.5 for labels: bead

Mask RCNN Precision: 1.0, Recall: 1.0 at IoU 0.5 for labels: cage
Mask RCNN F-1 Score: 1.0 at IoU 0.5 for labels: cage

Mask RCNN Precision: 0.8313500205170291, Recall: 0.9173647271904007 at IoU 0.5 for labels: hela-cell
Mask RCNN F-1 Score: 0.8722419545797008 at IoU 0.5 for labels: hela-cell


In [ ]:
cell_labels = np.where(np.array(all_labels) == 1)[0]
cell_features = [features[i] for i in cell_labels]
cell_types = [exp_types[i] for i in cell_labels]

In [ ]:
# filter filter cell types
types_to_consider = ['jurkat', 'hela-suspension']
idxs_to_keep: List[int] = []
for idx in range(len(cell_types)):
    if cell_types[idx] in types_to_consider:
        idxs_to_keep.append(idx)

cell_features = np.array([cell_features[i] for i in idxs_to_keep])
cell_types = [cell_types[i] for i in idxs_to_keep]

In [ ]:
# TSNE
from sklearn.manifold import TSNE
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
# we want to get T-SNE embedding with 2 dimensions
n_components = 2
tsne = TSNE(n_components)
tsne_result = tsne.fit_transform(cell_features)
# Plot the result of our TSNE with the label color coded
# A lot of the stuff here is about making the plot look pretty and not TSNE
tsne_result_df = pd.DataFrame({'tsne_1': tsne_result[:,0], 'tsne_2': tsne_result[:,1], 'label': cell_types})
fig, ax = plt.subplots(1)
sn.scatterplot(x='tsne_1', y='tsne_2', hue='label', data=tsne_result_df, ax=ax,s=120)
lim = (tsne_result.min()-5, tsne_result.max()+5)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
import umap
# UMAP embeddings will be with 2 dimensions
reducer = umap.UMAP()
umap_result = reducer.fit_transform(cell_features)
# Plot the result of our UMAP with the label color coded
umap_result_df = pd.DataFrame({'umap_1': umap_result[:,0], 'umap_2': umap_result[:,1], 'label': cell_types})
fig, ax = plt.subplots(1)
sn.scatterplot(x='umap_1', y='umap_2', hue='label', data=umap_result_df, ax=ax,s=120)
lim = (umap_result.min()-1, umap_result.max()+1)
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect('equal')
ax.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)

In [ ]:
img = cv2.imread('5_1_2_628_1_028675_005949_-16239_White.png', cv2.IMREAD_UNCHANGED)

In [ ]:
preds, run_time, debug_img = run_mask_rcnn(img, 
                                    normalize_image = True, 
                                    bit_depth = 12, 
                                    crop = True, 
                                    post_process_class_names = list(detector.get_label_map().values()),
                                    return_features=False,
                                    plot_results = True)

In [ ]:
Image.fromarray(debug_img)

In [ ]:
from PIL import Image